# Notebook 02 — Coleta do Senado Federal

**Sprint 1 — Lei e Política**

## Escopo deste notebook

O Senado é fonte de **enriquecimento** do corpus de temas. O MVP exige:

1. **Obrigatório:** Cadastro de senadores → `parlamentares` (casa='senado')
2. **Obrigatório:** Ementas de matérias do Senado → `proposicoes` (casa='senado') para enriquecer o corpus do clustering
3. **Stretch goal** (somente se o tempo permitir): votos nominais de senadores

**Por que enriquecer com o Senado?** O texto legislativo brasileiro circula pelas duas casas. Incluir ementas do Senado amplifica o corpus, melhora a cobertura dos temas e aumenta a diversidade semântica para o TF-IDF.

**Nota sobre IDs:** `id_externo` do Senado pode colidir com o da Câmara. O campo `casa='senado'` resolve isso — a restrição UNIQUE(id_externo, casa) garante integridade.

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
import re

from src.coleta import get_senado, salvar_raw, carregar_raw
from src.db import upsert_parlamentares, upsert_proposicoes

log = logging.getLogger('02_coleta_senado')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
print('Módulos carregados.')

## 1. Senadores em exercício

Endpoint: `GET /senador/lista/atual` → retorna os 81 senadores da legislatura atual.

In [ ]:
dados_senadores = get_senado('senador/lista/atual')
salvar_raw('senado_senadores.json', dados_senadores)

# A estrutura do JSON do Senado varia — inspecionar antes de parsear
print('Chaves de nível 1:', list(dados_senadores.keys()) if dados_senadores else 'ERRO — dados vazios')

In [ ]:
# Navegar no JSON até a lista de senadores
# Estrutura típica: ListaParlamentarEmExercicio > Parlamentares > Parlamentar
try:
    lista = (
        dados_senadores
        .get('ListaParlamentarEmExercicio', {})
        .get('Parlamentares', {})
        .get('Parlamentar', [])
    )
except AttributeError:
    lista = []

if not lista:
    # Tentar estrutura alternativa
    print('AVISO: estrutura padrão não encontrada. JSON bruto:')
    import json
    print(json.dumps(dados_senadores, ensure_ascii=False, indent=2)[:2000])
else:
    print(f'Senadores encontrados: {len(lista)}')
    print('Exemplo:', lista[0] if lista else 'N/A')

In [ ]:
def extrair_senador(s):
    """Extrai campos do dict de senador para o formato do banco."""
    ident = s.get('IdentificacaoParlamentar', {})
    mandato = s.get('Mandato', {})
    partido_obj = ident.get('SiglaPartidoParlamentar', '') or ident.get('SiglaPartido', '')
    uf = ident.get('UfParlamentar', '') or mandato.get('UfParlamentar', '')

    cod = ident.get('CodigoParlamentar') or ident.get('CodigoParlamenta')
    if not cod:
        return None

    return {
        'id_externo': int(cod),
        'casa': 'senado',
        'nome': ident.get('NomeParlamentar', '') or ident.get('NomeCompletoParlamentar', ''),
        'partido': str(partido_obj),
        'uf': str(uf),
        'foto_url': ident.get('UrlFotoParlamentar', ''),
    }

registros_sen = [r for s in lista if (r := extrair_senador(s)) is not None]
print(f'Senadores válidos para inserção: {len(registros_sen)}')

total = upsert_parlamentares(registros_sen)
print(f'Senadores inseridos/atualizados: {total}')

## 2. Ementas de matérias do Senado (corpus)

Endpoint de pesquisa de matérias. O Senado não tem paginação uniforme como a Câmara; usamos o endpoint de pesquisa com filtros de ano para controlar o volume.

In [ ]:
def data_segura(s):
    if not s:
        return None
    m = re.match(r'(\d{4}-\d{2}-\d{2})', str(s))
    return m.group(1) if m else None

materias_todas = []

for ano in [2023, 2024]:
    log.info('Coletando matérias do Senado — ano %d', ano)
    dados = get_senado(
        'materia/pesquisa/lista',
        params={
            'ano': ano,
            'indicadorVigencia': 'S',
        },
    )

    if not dados:
        log.warning('Nenhum dado retornado para ano %d', ano)
        continue

    # Inspecionar estrutura
    try:
        lista_mat = (
            dados
            .get('PesquisaBasicaMateria', {})
            .get('Materias', {})
            .get('Materia', [])
        )
    except AttributeError:
        log.warning('Estrutura inesperada para ano %d — salvando raw para inspeção', ano)
        salvar_raw(f'senado_materias_{ano}_raw.json', dados)
        lista_mat = []

    if isinstance(lista_mat, dict):   # quando há apenas 1 item, API retorna dict
        lista_mat = [lista_mat]

    materias_todas.extend(lista_mat)
    log.info('  ano %d: %d matérias', ano, len(lista_mat))

salvar_raw('senado_materias.json', materias_todas)
print(f'Total de matérias coletadas: {len(materias_todas)}')

In [ ]:
# Inspecionar estrutura de uma matéria para mapear campos
if materias_todas:
    import json
    print(json.dumps(materias_todas[0], ensure_ascii=False, indent=2))

In [ ]:
def extrair_proposicao_senado(m):
    """Extrai campos da matéria do Senado para o formato do banco."""
    ident = m.get('IdentificacaoMateria', {}) or m

    cod = ident.get('CodigoMateria')
    if not cod:
        return None

    ementa = (
        m.get('EmentaMateria')
        or ident.get('EmentaMateria')
        or m.get('Ementa', '')
        or ''
    ).strip()

    if len(ementa) < 10:
        return None

    data_str = (
        m.get('DataApresentacao')
        or ident.get('DataApresentacao')
        or ''
    )

    return {
        'id_externo': int(cod),
        'casa': 'senado',
        'ementa': ementa,
        'keywords': '',
        'data': data_segura(data_str),
    }

registros_mat = [r for m in materias_todas if (r := extrair_proposicao_senado(m)) is not None]
print(f'Matérias com ementa válida: {len(registros_mat)} / {len(materias_todas)}')

total_mat = upsert_proposicoes(registros_mat)
print(f'Proposições (senado) inseridas/atualizadas: {total_mat}')

## 3. Stretch goal — Votos nominais do Senado

> **Status:** implementar somente se o tempo da sprint permitir após conclusão dos itens obrigatórios.

O endpoint do Senado para votos nominais exige iterar pelas matérias e buscar sessões de votação. A estrutura é mais complexa que a da Câmara. Para o MVP, o perfil de votos cobre apenas deputados — o schema suporta a expansão futura via campo `casa`.

In [ ]:
# Stretch goal — descomente para implementar
#
# Para cada matéria do Senado:
# GET /materia/{codigo}/votacoes
# → iterar sessões → GET /votacao/{id}/votos
# Normalizar senador → parlamentar_id (casa='senado')
# Usar a mesma função normalizar_voto() de src/coleta.py
#
# IMPORTANTE: verificar o mapeamento de tipoVoto do Senado
# (pode diferir da Câmara) e adicionar ao MAPA_VOTO em src/coleta.py

print('Stretch goal do Senado: não implementado no MVP. Ver backlog no README.')

## 4. Sanidade final

In [ ]:
from src.db import get_client

client = get_client()

def contar(tabela, filtros=None):
    q = client.table(tabela).select('id', count='exact')
    if filtros:
        for col, val in filtros.items():
            q = q.eq(col, val)
    return q.execute().count

print('=== Sanidade do banco — após Sprint 1 (completa) ===')
print(f'parlamentares (câmara):  {contar("parlamentares", {"casa": "camara"})}')
print(f'parlamentares (senado):  {contar("parlamentares", {"casa": "senado"})}')
print(f'proposições (câmara):    {contar("proposicoes", {"casa": "camara"})}')
print(f'proposições (senado):    {contar("proposicoes", {"casa": "senado"})}')
print(f'votações:                {contar("votacoes")}')
print(f'votos:                   {contar("votos")}')
print(f'  favoravel:             {contar("votos", {"voto": "favoravel"})}')
print(f'  contrario:             {contar("votos", {"voto": "contrario"})}')
print(f'  abstencao:             {contar("votos", {"voto": "abstencao"})}')